In [5]:
import scvi
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
sc.set_figure_params(figsize=(8, 8))

In [6]:
adata = sc.read_h5ad('/rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_blood/code/rna/mapping/scVI_test/anndata.h5ad')

In [7]:
adata

AnnData object with n_obs × n_vars = 147203 × 15403
    obs: 'sample', 'stage', 'celltype', 'origin'
    var: 'name'

In [8]:
# preprocessing
sc.pp.filter_genes(adata, min_counts=3)
adata.layers["counts"] = adata.X.copy() # preserve counts
adata.raw = adata # freeze the state in `.raw`

sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2500,
    subset=True,
    layer="counts",
    flavor="cell_ranger",
    batch_key="stage"
)

/home/bt392/miniconda3/envs/env_scvi2/lib/python3.9/site-packages/scanpy/preprocessing/_highly_variable_genes.py:475: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  hvg = hvg.append(missing_hvg, ignore_index=True)
/home/bt392/miniconda3/envs/env_scvi2/lib/python3.9/site-packages/scanpy/preprocessing/_highly_variable_genes.py:475: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  hvg = hvg.append(missing_hvg, ignore_index=True)
/home/bt392/miniconda3/envs/env_scvi2/lib/python3.9/site-packages/scanpy/preprocessing/_highly_variable_genes.py:475: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  hvg = hvg.append(missing_hvg, ignore_index=True)
/home/bt392/miniconda3/envs/env_scvi2/lib/python3.9/site-packages/scanpy/preprocessing/_highly

#### Figure out how to do the batch/covariate correction
how to compare stage with day of in vitro
what are the batches vs covariates? 
maybe if I make the day of in vitro a stage column it will automatically batch correct both the origin and the timepoint
|

# Test 1

In [9]:
scvi.model.SCVI.setup_anndata(
    adata,
    layer="counts",
    batch_key="stage",
    categorical_covariate_keys=["sample"]
)

INFO     Using batches from adata.obs["stage"]                                               
INFO     No label_key inputted, assuming all cells have same label                           
INFO     Using data from adata.layers["counts"]                                              
INFO     Successfully registered anndata object containing 147203 cells, 2500 vars, 14       
         batches, 1 labels, and 0 proteins. Also registered 1 extra categorical covariates   
         and 0 extra continuous covariates.                                                  
INFO     Please do not further modify adata until model is trained.                          


/home/bt392/miniconda3/envs/env_scvi2/lib/python3.9/site-packages/scvi/data/_anndata.py:315: UserWarning: Training will be faster when sparse matrix is formatted as CSR. It is safe to cast before model initialization.
  warnings.warn(


In [10]:
model = scvi.model.SCVI(adata, n_latent=40)

In [ ]:
model.train()

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
Set SLURM handle signals.


Epoch 1/54:   0%|          | 0/54 [00:00<?, ?it/s]

In [ ]:
model.save("/rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_bloodcode/rna/mapping/scVI_test/scvi_model_try1/")

In [ ]:
latent = model.get_latent_representation()
adata.obsm["X_scVI"] = latent
adata.layers["scvi_normalized"] = model.get_normalized_expression(
    library_size=10e4
)

In [ ]:
adata.obs.tail()

In [ ]:
# use scVI latent space for UMAP generation
sc.pp.neighbors(adata, use_rep="X_scVI", n_neighbors=35)
sc.tl.umap(adata, min_dist=0.4)

In [ ]:
sc.pl.umap(
    adata,
    color=["celltype", "origin", "stage", "sample"],
    frameon=False,
    ncols=2, legend_loc='none',
    save = 'try1.png'
)

# Test 2

In [ ]:
scvi.model.SCVI.setup_anndata(
    adata,
    layer="counts",
    batch_key="origin",
    categorical_covariate_keys=["stage", "sample"]
)

In [ ]:
model = scvi.model.SCVI(adata, n_latent=40)

In [ ]:
model.train()

In [ ]:
model.save("/rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_bloodcode/rna/mapping/scVI_test/scvi_model_try2/")

In [ ]:
latent = model.get_latent_representation()
adata.obsm["X_scVI"] = latent
adata.layers["scvi_normalized"] = model.get_normalized_expression(
    library_size=10e4
)

In [ ]:
adata.obs.tail()

In [ ]:
# use scVI latent space for UMAP generation
sc.pp.neighbors(adata, use_rep="X_scVI", n_neighbors=35)
sc.tl.umap(adata, min_dist=0.4)

In [ ]:
sc.pl.umap(
    adata,
    color=["celltype", "origin", "stage", "sample"],
    frameon=False,
    ncols=2, legend_loc='none',
    save = 'try2.png'
)

# Test 3

In [ ]:
scvi.model.SCVI.setup_anndata(
    adata,
    layer="counts",
    batch_key="sample",
    categorical_covariate_keys=["stage", "origin"]
)

In [ ]:
model = scvi.model.SCVI(adata, n_latent=40)

In [ ]:
model.train()

In [ ]:
model.save("/rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_bloodcode/rna/mapping/scVI_test/scvi_model_try3/")

In [ ]:
latent = model.get_latent_representation()
adata.obsm["X_scVI"] = latent
adata.layers["scvi_normalized"] = model.get_normalized_expression(
    library_size=10e4
)

In [ ]:
adata.obs.tail()

In [ ]:
# use scVI latent space for UMAP generation
sc.pp.neighbors(adata, use_rep="X_scVI", n_neighbors=35)
sc.tl.umap(adata, min_dist=0.4)

In [ ]:
sc.pl.umap(
    adata,
    color=["celltype", "origin", "stage", "sample"],
    frameon=False,
    ncols=2, legend_loc='none',
    save = 'try3.png'
)